In [2]:
import os
import logging
import pandas as pd
import numpy as np

# Configure Logging for Auditing & Traceability
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def run_data_cleaning_pipeline():
    logging.info("Starting Enterprise Data Cleaning Pipeline...")
    
    # Ensure target output directory exists
    processed_dir = '../data/processed'
    os.makedirs(processed_dir, exist_ok=True)
    
    # ==========================================
    # STEP 1: LOAD RAW DATASETS
    # ==========================================
    logging.info("Step 1: Loading raw datasets...")
    orders = pd.read_csv('../data/olist_orders_dataset.csv')
    items = pd.read_csv('../data/olist_order_items_dataset.csv')
    products = pd.read_csv('../data/olist_products_dataset.csv')
    payments = pd.read_csv('../data/olist_order_payments_dataset.csv')
    reviews = pd.read_csv('../data/olist_order_reviews_dataset.csv')
    customers = pd.read_csv('../data/olist_customers_dataset.csv')
    sellers = pd.read_csv('../data/olist_sellers_dataset.csv')
    translations = pd.read_csv('../data/product_category_name_translation.csv')

    # ==========================================
    # STEP 2: DEDUPLICATION
    # ==========================================
    logging.info("Step 2: Checking and removing duplicates...")
    datasets = {
        'orders': orders, 'items': items, 'products': products,
        'payments': payments, 'reviews': reviews, 
        'customers': customers, 'sellers': sellers
    }
    
    for name, df in datasets.items():
        initial_count = len(df)
        df.drop_duplicates(inplace=True)
        dedup_count = len(df)
        if initial_count - dedup_count > 0:
            logging.warning(f"Removed {initial_count - dedup_count} exact duplicate rows from {name}.")
        else:
            logging.info(f"No duplicate rows found in {name}.")

    # ==========================================
    # STEP 3: DATATYPE CASTING & FORMATTING
    # ==========================================
    logging.info("Step 3: Casting explicit datatypes and formatting strings...")
    
    # 3a. Datetime Parsing for Orders
    datetime_cols = [
        'order_purchase_timestamp', 'order_approved_at',
        'order_delivered_carrier_date', 'order_delivered_customer_date',
        'order_estimated_delivery_date'
    ]
    for col in datetime_cols:
        orders[col] = pd.to_datetime(orders[col], errors='coerce')
        
    # 3b. String Normalization (Lowercasing & Whitespace Stripping)
    customers['customer_city'] = customers['customer_city'].str.strip().str.lower()
    customers['customer_state'] = customers['customer_state'].str.strip().str.upper()
    sellers['seller_city'] = sellers['seller_city'].str.strip().str.lower()
    sellers['seller_state'] = sellers['seller_state'].str.strip().str.upper()

    # ==========================================
    # STEP 4: CATEGORICAL MAPPING & MISSING VALUE IMPUTATION
    # ==========================================
    logging.info("Step 4: Merging category translations and imputing missing values...")
    
    # 4a. Map Portuguese Categories to English
    products = products.merge(translations, on='product_category_name', how='left')
    products['product_category_name_english'] = products['product_category_name_english'].fillna('unknown')
    products.drop(columns=['product_category_name'], inplace=True)
    
    # 4b. Impute Missing Review Text
    reviews['review_comment_title'] = reviews['review_comment_title'].fillna('No Title').str.strip()
    reviews['review_comment_message'] = reviews['review_comment_message'].fillna('No Comment Provided').str.strip()
    
    # 4c. Physical Product Metadata Imputation (Median Imputation for numerical metrics)
    product_num_cols = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
    for col in product_num_cols:
        median_val = products[col].median()
        products[col] = products[col].fillna(median_val)

    # ==========================================
    # STEP 5: FEATURE ENGINEERING & METRICS
    # ==========================================
    logging.info("Step 5: Engineering business delivery metrics...")
    
    # Calculate delivery delay in days (Actual Delivery - Estimated Delivery)
    orders['delivery_delay_days'] = (
        orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']
    ).dt.total_seconds() / (24 * 3600)

    # Boolean indicator: True if actual delivery exceeded promised SLA date
    orders['is_delayed'] = orders['delivery_delay_days'] > 0

    # Total fulfillment lead time in days
    orders['delivery_time_days'] = (
        orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
    ).dt.total_seconds() / (24 * 3600)

    # Temporal features for trend analysis
    orders['purchase_year'] = orders['order_purchase_timestamp'].dt.year
    orders['purchase_month'] = orders['order_purchase_timestamp'].dt.to_period('M').astype(str)
    orders['purchase_dayofweek'] = orders['order_purchase_timestamp'].dt.day_name()

    # ==========================================
    # STEP 6: DATA VALIDATION & CONSISTENCY CHECKS
    # ==========================================
    logging.info("Step 6: Executing automated data validation rules...")
    
    # Rule 1: Delivered orders should not have a delivery date prior to purchase date
    invalid_dates = orders[
        (orders['order_delivered_customer_date'] < orders['order_purchase_timestamp']) &
        (orders['order_status'] == 'delivered')
    ]
    if len(invalid_dates) > 0:
        logging.error(f"DATA INTEGRITY ERROR: Found {len(invalid_dates)} orders delivered prior to purchase!")
    else:
        logging.info("Validation Passed: Zero temporal paradoxes in delivery dates.")

    # Rule 2: Order items price must be non-negative
    invalid_prices = items[items['price'] < 0]
    if len(invalid_prices) > 0:
        logging.error(f"DATA INTEGRITY ERROR: Found {len(invalid_prices)} items with negative price!")
    else:
        logging.info("Validation Passed: All product item prices are positive.")

    # ==========================================
    # STEP 7: EXPORT CLEAN PRODUCTION DATASETS
    # ==========================================
    logging.info("Step 7: Exporting clean datasets to data/processed/...")
    
    orders.to_csv(os.path.join(processed_dir, 'clean_orders.csv'), index=False)
    items.to_csv(os.path.join(processed_dir, 'clean_order_items.csv'), index=False)
    products.to_csv(os.path.join(processed_dir, 'clean_products.csv'), index=False)
    payments.to_csv(os.path.join(processed_dir, 'clean_payments.csv'), index=False)
    reviews.to_csv(os.path.join(processed_dir, 'clean_reviews.csv'), index=False)
    customers.to_csv(os.path.join(processed_dir, 'clean_customers.csv'), index=False)
    sellers.to_csv(os.path.join(processed_dir, 'clean_sellers.csv'), index=False)

    logging.info("SUCCESS: Production Data Cleaning Pipeline executed successfully!")

# Run Pipeline
if __name__ == "__main__":
    run_data_cleaning_pipeline()

2026-07-24 12:26:48,669 - INFO - Starting Enterprise Data Cleaning Pipeline...
2026-07-24 12:26:48,686 - INFO - Step 1: Loading raw datasets...
2026-07-24 12:26:50,599 - INFO - Step 2: Checking and removing duplicates...
2026-07-24 12:26:51,138 - INFO - No duplicate rows found in orders.
2026-07-24 12:26:51,408 - INFO - No duplicate rows found in items.
2026-07-24 12:26:51,428 - INFO - No duplicate rows found in products.
2026-07-24 12:26:51,488 - INFO - No duplicate rows found in payments.
2026-07-24 12:26:51,986 - INFO - No duplicate rows found in reviews.
2026-07-24 12:26:52,064 - INFO - No duplicate rows found in customers.
2026-07-24 12:26:52,069 - INFO - No duplicate rows found in sellers.
2026-07-24 12:26:52,070 - INFO - Step 3: Casting explicit datatypes and formatting strings...
2026-07-24 12:26:52,356 - INFO - Step 4: Merging category translations and imputing missing values...
2026-07-24 12:26:52,433 - INFO - Step 5: Engineering business delivery metrics...
2026-07-24 12:26: